# Teardown VM — proj03

Detaches the block volume **(preserved)**, then deletes the VM and lease on **KVM@TACC**.

⚠️ **Does NOT delete:**
- `block-<username>-proj03` — Cinder volume (data survives)
- `photoprism-proj03` — Swift container on CHI@TACC

Safe to re-run — handles already-deleted resources gracefully.

In [ ]:
from chi import server, context, lease
import chi, os

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
project = "proj03"

print(f"User: {username} | Project: {project} | Site: KVM@TACC")

In [ ]:
# Detach block volume (preserve it — do NOT delete)
try:
    cinder_client = chi.clients.cinder()
    volumes = [v for v in cinder_client.volumes.list() if v.name == f"block-{username}-{project}"]
    if volumes:
        volume = volumes[0]
        if volume.status == "in-use":
            s = server.get_server(f"node-{username}-{project}")
            s.detach_volume(volume.id)
            print(f"Block volume '{volume.name}' detached (preserved).")
        else:
            print(f"Block volume '{volume.name}' already detached.")
    else:
        print("No block volume found — skipping detach.")
except Exception as e:
    print(f"Could not detach block volume: {e}")

In [ ]:
# Delete the VM server
try:
    s = server.get_server(f"node-{username}-{project}")
    server.delete_server(s.id)
    print(f"Server node-{username}-{project} deleted.")
except Exception as e:
    print(f"Server not found or already deleted: {e}")

In [ ]:
# Release the lease
try:
    l = lease.get_lease(f"lease-{username}-{project}")
    lease.delete_lease(l.id)
    print(f"Lease lease-{username}-{project} deleted.")
except Exception as e:
    print(f"Lease not found or already deleted: {e}")

print("\nTeardown complete. Block volume and object storage intact.")